<a href="https://colab.research.google.com/github/qurban-12/ML_internship_Qurban/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qurban-12/ML_internship_Qurban/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Growing vs. declining content

The paper reports a difference between growing and declining content, including a difference in average content age. My methodology question would be: **How exactly was the growing/declining label constructed, and is the label based on a future outcome or on measurements from the same observation window?** If the label and the measured features come from the same window, I would want to understand how the design avoids using information that would only be known after the prediction point.

This is a constructive question because the interpretation depends on when the label becomes known. A clearly defined prediction window and label-generation rule would make the finding easier to reproduce and interpret.

### Finding 2: Content performance by age

The paper reports that content performance varies across age windows, including a reported peak around the 61–90 day range. My methodology question would be: **Does the validation or cohort design support interpreting this pattern as a directional association, or could differences between age groups reflect differences in the types of pages entering each group?**

I would also check whether the age variable and outcome are measured independently and whether the analysis controls for repeated pages or clients. This would help distinguish an observed cohort pattern from a stronger claim about what causes content performance to change.


In [47]:
!git clone https://github.com/qurban-12/ML_internship_Qurban.git

fatal: destination path 'ML_internship_Qurban' already exists and is not an empty directory.


In [48]:
!ls /content/ML_internship_Qurban

AGENTS.md    docs		   notebooks	     scripts	 work
CLAUDE.md    GUIDE.md		   outputs	     SETUP.md
data	     LICENSE		   README.md	     skills
DATA_USE.md  ML_internship_Qurban  requirements.txt  submission


In [49]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys

os.chdir("/content/ML_internship_Qurban")
sys.path.insert(0, "/content/ML_internship_Qurban")

print("Working directory:", os.getcwd())

Working directory: /content/ML_internship_Qurban


In [50]:
from scripts.ml_utils import precision_at_k

print("✅ precision_at_k imported successfully")

✅ precision_at_k imported successfully


In [51]:
import os

print(os.path.exists(
    "/content/ML_internship_Qurban/data/raw/content_refresh_anonymized.csv"
))

print(os.path.exists(
    "/content/ML_internship_Qurban/scripts/ml_utils.py"
))

True
True


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [52]:
print("Dataset columns:")
for i, col in enumerate(df.columns, 1):
    print(i, col)

Dataset columns:
1 content_id
2 client_id
3 search_volume
4 competition
5 competition_level
6 cpc
7 content_type
8 main_intent
9 word_count
10 char_count
11 provider_used
12 model_used
13 impressions_90d
14 clicks_90d
15 pageviews_90d
16 sessions_90d
17 users_90d
18 engaged_sessions_90d
19 ai_sessions_90d
20 scroll_events_90d
21 days_with_impressions
22 days_with_sessions
23 impressions_last_30d
24 clicks_last_30d
25 sessions_last_30d
26 impressions_prev_30d
27 clicks_prev_30d
28 sessions_prev_30d
29 content_age_days
30 age_tier
31 age_tier_order
32 days_since_last_update
33 freshness_tier
34 word_count_tier
35 char_count_tier
36 ctr
37 avg_position
38 engagement_rate
39 scroll_rate
40 ai_traffic_pct
41 impression_tier
42 position_tier
43 trend_direction
44 trend_pct
45 is_declining_label
46 log_impressions_90d
47 log_clicks_90d
48 log_sessions_90d
49 log_ai_sessions_90d
50 has_clicks
51 has_ai_sessions
52 measurable_opportunity


In [53]:
print("\nWeek-5 log feature source columns:")
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    print(col, "->", col in df.columns)


Week-5 log feature source columns:
impressions_90d -> True
clicks_90d -> True
sessions_90d -> True
ai_sessions_90d -> True


In [54]:
print(df[[
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d"
]].head())

   impressions_90d  clicks_90d  sessions_90d  ai_sessions_90d
0             3803          29            17                0
1            15320           7             9                0
2            12581          11            11                0
3            11751          58            78                0
4            19140          24           145                0


In [55]:
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

In [56]:
print("Dataset columns:")
for i, col in enumerate(df.columns, 1):
    print(i, col)

print("\nWeek-5 log feature source columns:")
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    print(col, "->", col in df.columns)

Dataset columns:
1 content_id
2 client_id
3 search_volume
4 competition
5 competition_level
6 cpc
7 content_type
8 main_intent
9 word_count
10 char_count
11 provider_used
12 model_used
13 impressions_90d
14 clicks_90d
15 pageviews_90d
16 sessions_90d
17 users_90d
18 engaged_sessions_90d
19 ai_sessions_90d
20 scroll_events_90d
21 days_with_impressions
22 days_with_sessions
23 impressions_last_30d
24 clicks_last_30d
25 sessions_last_30d
26 impressions_prev_30d
27 clicks_prev_30d
28 sessions_prev_30d
29 content_age_days
30 age_tier
31 age_tier_order
32 days_since_last_update
33 freshness_tier
34 word_count_tier
35 char_count_tier
36 ctr
37 avg_position
38 engagement_rate
39 scroll_rate
40 ai_traffic_pct
41 impression_tier
42 position_tier
43 trend_direction
44 trend_pct
45 is_declining_label
46 log_impressions_90d
47 log_clicks_90d
48 log_sessions_90d
49 log_ai_sessions_90d
50 has_clicks
51 has_ai_sessions
52 measurable_opportunity

Week-5 log feature source columns:
impressions_90d -> Tr

In [57]:
# Verify the exact Week-5 feature set

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

target_col = "is_declining_label"
group_col = "client_id"

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numeric_features) + len(categorical_features))

print("\nMissing numeric features:")
print([c for c in numeric_features if c not in df.columns])

print("\nMissing categorical features:")
print([c for c in categorical_features if c not in df.columns])

print("\nTarget exists:", target_col in df.columns)
print("Group exists:", group_col in df.columns)

Numeric features: 18
Categorical features: 8
Total features: 26

Missing numeric features:
[]

Missing categorical features:
[]

Target exists: True
Group exists: True


In [58]:
# Recreate the exact Week-5 prepared dataset

import subprocess
from pathlib import Path

repo_path = Path("/content/ML_internship_Qurban")

result = subprocess.run(
    ["python", "scripts/01_prepare_features.py"],
    cwd=repo_path,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

PREPARED_PATH = repo_path / "data/processed/refresh_feature_vector.csv"

print("Prepared file exists:", PREPARED_PATH.exists())

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/ML_internship_Qurban/data/processed/refresh_feature_vector.csv

Prepared file exists: True


In [59]:
# Load the exact Week-5 prepared dataset

df = pd.read_csv(PREPARED_PATH)

print("Prepared dataset shape:", df.shape)
print("Target exists:", "is_declining_label" in df.columns)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(round(df["is_declining_label"].mean() * 100, 2), "%")

Prepared dataset shape: (30000, 52)
Target exists: True

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
54.21 %


In [60]:
# Check client distribution before splitting

print("Number of unique clients:", df["client_id"].nunique())

print("\nRows per client:")
print(df["client_id"].value_counts().describe())

print("\nDeclining rate by client:")
client_rates = (
    df.groupby("client_id")["is_declining_label"]
      .agg(["count", "mean"])
      .sort_values("mean")
)

print(client_rates)

Number of unique clients: 32

Rows per client:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: count, dtype: float64

Declining rate by client:
                   count      mean
client_id                         
client_1a6562590e      3  0.000000
client_25fc0e7096    476  0.000000
client_e29c9c180c    708  0.086158
client_2c624232cd    649  0.194145
client_9f14025af0    325  0.196923
client_d4735e3a26   1106  0.219711
client_8b940be7fb     28  0.321429
client_b4944c6ff0    254  0.385827
client_d59eced1de     43  0.418605
client_02d20bbd7e     38  0.421053
client_8527a891e2   1194  0.437186
client_e629fa6598    720  0.455556
client_0b918943df     35  0.485714
client_19581e27de   7008  0.490154
client_4e07408562   2294  0.493461
client_434c9b5ae5     87  0.517241
client_4fc82b26ae     32  0.531250
client_f74efabef1   1031  0.542192
client_bbb965ab0c    505  0.54

In [61]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, roc_auc_score


# Prepare X and y

X = df[numeric_features + categorical_features].copy()
y = df[target_col].astype(int)
groups = df[group_col]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])


# 1. RANDOM ROW SPLIT

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model.fit(X_train_r, y_train_r)
prob_random = model.predict_proba(X_test_r)[:, 1]

# Precision at K
def precision_at_k_local(y_true, scores, k):
    k = min(k, len(scores))
    top_idx = scores.argsort()[::-1][:k]
    return y_true.iloc[top_idx].mean()

random_p20 = precision_at_k_local(y_test_r.reset_index(drop=True), prob_random, 20)
random_p50 = precision_at_k_local(y_test_r.reset_index(drop=True), prob_random, 50)
random_p100 = precision_at_k_local(y_test_r.reset_index(drop=True), prob_random, 100)

random_auc = roc_auc_score(y_test_r, prob_random)


# 2. HONEST CLIENT-GROUPED SPLIT

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]
y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

model.fit(X_train_g, y_train_g)
prob_grouped = model.predict_proba(X_test_g)[:, 1]

grouped_p20 = precision_at_k_local(
    y_test_g.reset_index(drop=True),
    prob_grouped,
    20
)

grouped_p50 = precision_at_k_local(
    y_test_g.reset_index(drop=True),
    prob_grouped,
    50
)

grouped_p100 = precision_at_k_local(
    y_test_g.reset_index(drop=True),
    prob_grouped,
    100
)

grouped_auc = roc_auc_score(y_test_g, prob_grouped)


# REPORT

print("===== RANDOM ROW SPLIT =====")
print("Train rows:", len(X_train_r))
print("Test rows:", len(X_test_r))
print("Shared clients:", len(set(groups.iloc[X_train_r.index]) & set(groups.iloc[X_test_r.index])))
print("Test declining rate:", round(y_test_r.mean() * 100, 2), "%")
print("ROC-AUC:", round(random_auc, 4))
print("P@20:", round(random_p20, 3))
print("P@50:", round(random_p50, 3))
print("P@100:", round(random_p100, 3))

print("\n===== HONEST CLIENT-GROUPED SPLIT =====")
print("Train rows:", len(X_train_g))
print("Test rows:", len(X_test_g))
print("Train clients:", len(set(train_groups)))
print("Test clients:", len(set(test_groups)))
print("Shared clients:", len(set(train_groups) & set(test_groups)))
print("Test declining rate:", round(y_test_g.mean() * 100, 2), "%")
print("ROC-AUC:", round(grouped_auc, 4))
print("P@20:", round(grouped_p20, 3))
print("P@50:", round(grouped_p50, 3))
print("P@100:", round(grouped_p100, 3))

===== RANDOM ROW SPLIT =====
Train rows: 24000
Test rows: 6000
Shared clients: 31
Test declining rate: 54.2 %
ROC-AUC: 0.7106
P@20: 0.9
P@50: 0.92
P@100: 0.89

===== HONEST CLIENT-GROUPED SPLIT =====
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Shared clients: 0
Test declining rate: 51.1 %
ROC-AUC: 0.6157
P@20: 0.7
P@50: 0.72
P@100: 0.7


### 2. My model under honest split (before/after)

I compared a random row split with a client-grouped split.

| Metric              | Random row split | Honest client-grouped split |
| ------------------- | ---------------: | --------------------------: |
| Train rows          |           24,000 |                      23,837 |
| Test rows           |            6,000 |                       6,163 |
| Shared clients      |               31 |                           0 |
| Test declining rate |            54.2% |                       51.1% |
| ROC-AUC             |           0.7106 |                      0.6157 |
| P@20                |            0.900 |                       0.700 |
| P@50                |            0.920 |                       0.720 |
| P@100               |            0.890 |                       0.700 |

The random split produced higher measured performance, but 31 clients were present in both the training and test sets. The client-grouped split kept all clients separated, with 25 clients in training and 7 clients in testing.

The grouped evaluation therefore provides a stricter estimate of performance when applying the model to clients that were not represented during training. The drop in measured performance from the random split to the grouped split is a validation gap that should be reported rather than hidden.

For this dataset, I will use the client-grouped evaluation when describing model performance. I will describe the model as providing ranking or decision-support evidence for the observed declining-label task rather than claiming that it reliably predicts future decline for unseen clients.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [62]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# SECTION 3 — LEAKAGE AUDIT


print("===== LEAKAGE AUDIT =====")


# 1. Confirm forbidden label-derived columns are excluded

forbidden_features = [
    "trend_direction",
    "trend_pct"
]

print("\n1. Label-derived columns")
for col in forbidden_features:
    print(
        f"{col}:",
        "IN FEATURES" if col in numeric_features + categorical_features
        else "excluded"
    )


# 2. Check whether any model feature is exactly identical
#    to the target

print("\n2. Features identical to target")

identical_features = []

for col in numeric_features + categorical_features:
    try:
        if df[col].equals(df[target_col]):
            identical_features.append(col)
    except Exception:
        pass

print("Identical features:", identical_features)


# 3. Check feature-target relationship using simple
#    one-feature correlations for numeric features

print("\n3. Numeric feature correlations with target")

correlations = []

for col in numeric_features:
    corr = df[[col, target_col]].corr(numeric_only=True).iloc[0, 1]
    correlations.append((col, corr))

correlations = sorted(
    correlations,
    key=lambda x: abs(x[1]) if pd.notna(x[1]) else 0,
    reverse=True
)

for col, corr in correlations:
    print(f"{col:30s} {corr:.4f}")


# 4. Check feature availability / missingness

print("\n4. Missingness audit")

missing_summary = (
    df[numeric_features + categorical_features]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print(missing_summary)


# 5. Check suspicious feature families

print("\n5. Potentially sensitive feature families")

sensitive_groups = {
    "trend/label-related": [
        "trend_direction",
        "trend_pct"
    ],
    "recent-period metrics": [
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d"
    ],
    "previous-period metrics": [
        "impressions_prev_30d",
        "clicks_prev_30d",
        "sessions_prev_30d"
    ],
    "90-day metrics": [
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ai_sessions_90d"
    ]
}

for group_name, cols in sensitive_groups.items():
    present = [c for c in cols if c in numeric_features + categorical_features]
    print(f"{group_name}: {present}")


# 6. Final feature list

print("\n6. Final model features")

for i, col in enumerate(numeric_features + categorical_features, 1):
    print(f"{i:2d}. {col}")

print("\n===== END LEAKAGE AUDIT =====")

===== LEAKAGE AUDIT =====

1. Label-derived columns
trend_direction: excluded
trend_pct: excluded

2. Features identical to target
Identical features: []

3. Numeric feature correlations with target
days_with_impressions          0.1901
log_impressions_90d            0.1775
content_age_days               -0.1639
word_count                     0.1189
char_count                     0.1080
days_since_last_update         0.0814
ctr                            -0.0619
avg_position                   -0.0290
days_with_sessions             -0.0251
log_sessions_90d               0.0153
search_volume                  -0.0138
engagement_rate                -0.0127
competition                    0.0126
cpc                            -0.0060
log_ai_sessions_90d            -0.0043
log_clicks_90d                 0.0035
scroll_rate                    -0.0027
ai_traffic_pct                 0.0024

4. Missingness audit
search_volume             0.0
competition               0.0
cpc                       

In [63]:

# CHECK EXACT LABEL CONSTRUCTION


print("Target distribution:")
print(df[target_col].value_counts())

print("\nTarget mean:", round(df[target_col].mean(), 4))

print("\nTrend direction distribution in prepared data:")
if "trend_direction" in df.columns:
    print(df["trend_direction"].value_counts(dropna=False))
else:
    print("trend_direction is not present in prepared dataset.")

print("\nTrend percentage summary:")
if "trend_pct" in df.columns:
    print(df["trend_pct"].describe())
else:
    print("trend_pct is not present in prepared dataset.")

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target mean: 0.5421

Trend direction distribution in prepared data:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend percentage summary:
count    30000.000000
mean        -4.245473
std        446.304629
min       -100.000000
25%        -58.700000
50%        -26.000000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64


In [64]:

# INSPECT THE PREPARATION SCRIPT
from pathlib import Path

script_path = Path(
    "/content/ML_internship_Qurban/scripts/01_prepare_features.py"
)

print("Script exists:", script_path.exists())

if script_path.exists():
    print("\n===== PREPARATION SCRIPT =====\n")
    print(script_path.read_text())

Script exists: True

===== PREPARATION SCRIPT =====

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_utils import (
    BOOLEAN_COLUMNS,
    CATEGORICAL_COLUMNS,
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    NUMERIC_COLUMNS,
    PROCESSED_DIR,
    RAW_PATH,
    display_path,
    ensure_dirs,
    to_bool_series,
    write_json,
)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Prepare FlyRank refresh feature vector.")
    parser.add_argument("--input", default=str(RAW_PATH), help="Raw anonymized CSV export.")
    parser.add_argument(
        "--output",
        default=str(PROCESSED_DIR / "refresh_feature_vector.csv"),
        help="Prepared feature-vector CSV.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    ensure_dirs()

    input_path = Path(args.input)
    if not input_path.exists():
        raise FileNot

In [65]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

In [66]:
from pathlib import Path

repo = Path("/content/ML_internship_Qurban")

# Search project documentation and scripts for trend definitions
matches = []

for path in list(repo.rglob("*.md")) + list(repo.rglob("*.py")) + list(repo.rglob("*.sql")):
    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    if "trend_direction" in text or "trend_pct" in text:
        matches.append(path)

print("Files mentioning trend_direction/trend_pct:")
for path in matches:
    print("-", path.relative_to(repo))

Files mentioning trend_direction/trend_pct:
- DATA_USE.md
- GUIDE.md
- work/capstone_report_template.md
- docs/ml-intern-dataset-and-lane-guide.md
- docs/data-dictionary.md
- ML_internship_Qurban/DATA_USE.md
- ML_internship_Qurban/GUIDE.md
- ML_internship_Qurban/work/capstone_report_template.md
- ML_internship_Qurban/docs/ml-intern-dataset-and-lane-guide.md
- ML_internship_Qurban/docs/data-dictionary.md
- ML_internship_Qurban/skills/flyrank/flyrank-data/SKILL.md
- skills/flyrank/flyrank-data/SKILL.md
- scripts/04_evaluate_and_export.py
- scripts/05_build_pdf_report.py
- scripts/ml_utils.py
- scripts/01_prepare_features.py
- scripts/02_baseline_score.py
- ML_internship_Qurban/scripts/04_evaluate_and_export.py
- ML_internship_Qurban/scripts/05_build_pdf_report.py
- ML_internship_Qurban/scripts/ml_utils.py
- ML_internship_Qurban/scripts/01_prepare_features.py
- ML_internship_Qurban/scripts/02_baseline_score.py


In [67]:
# Print the relevant lines from the data dictionary

data_dict = repo / "docs/data-dictionary.md"

print("Data dictionary exists:", data_dict.exists())

if data_dict.exists():
    lines = data_dict.read_text(errors="ignore").splitlines()

    for i, line in enumerate(lines):
        if "trend_direction" in line.lower() or "trend_pct" in line.lower():
            start = max(0, i - 3)
            end = min(len(lines), i + 6)

            print("\n--- Relevant section ---")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")

Data dictionary exists: True

--- Relevant section ---
7: ## Read this first — the three rules that prevent 90% of mistakes
8: 
9: 1. **Rate columns are ×100 percentages.** `ctr = 0.76` means **0.76%**, not 76%. Applies to
10:    `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
11: 2. **The label comes from `trend_direction`.** The pipeline defines
12:    `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
13:    must **never** be model features — that's the leakage notebook 02 demonstrates.
14: 3. **IDs are for grouping only.** `content_id` / `client_id` are pseudonyms: use them for
15:    joins and grouped train/test splits, never as features.

--- Relevant section ---
8: 
9: 1. **Rate columns are ×100 percentages.** `ctr = 0.76` means **0.76%**, not 76%. Applies to
10:    `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
11: 2. **The label comes from `trend_direction`.** The pipeline defines
12:    `is

### 3. Leakage audit

The target is defined as `is_declining_label = (trend_direction == "down")`. The data dictionary defines `trend_direction` using the change in impressions between the last 30 days and the previous 30 days, with `down` corresponding to a decrease greater than 20%.

I explicitly excluded `trend_direction` and `trend_pct` from the 26 model features because they directly define or quantify the target. Including either column would constitute direct label leakage.

The model features contain no direct copy of the target, and the final feature list contains no `trend_direction` or `trend_pct`.

However, the audit identified an important temporal limitation. Several model inputs are 90-day aggregate metrics, while the label is calculated from recent 30-day and previous-30-day impression windows. Therefore, these aggregate features can overlap the time periods used to construct the observed decline label. This does not by itself establish direct leakage, but it limits how the model should be interpreted.

For this reason, I will avoid describing the current model as a validated future-prediction system. The safer interpretation is that it ranks content associated with the observed declining label and can provide decision-support evidence.

A stronger future-prediction evaluation would require a time-aware design in which all features are measured strictly before the period used to define the future outcome.


In [68]:

# SECTION 3 — DELIBERATE LEAKAGE TEST


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Use the honest client-grouped split created earlier
X_clean = df[numeric_features + categorical_features].copy()
y_clean = df[target_col].astype(int)

# Reuse the same grouped indices
X_train_clean = X_clean.iloc[train_idx]
X_test_clean = X_clean.iloc[test_idx]
y_train_clean = y_clean.iloc[train_idx]
y_test_clean = y_clean.iloc[test_idx]


# Clean baseline

clean_model = model
clean_model.fit(X_train_clean, y_train_clean)

clean_prob = clean_model.predict_proba(X_test_clean)[:, 1]
clean_auc = roc_auc_score(y_test_clean, clean_prob)


# Deliberately leaky feature:
# the target itself

X_leaky = X_clean.copy()
X_leaky["DELIBERATE_LEAK_TARGET"] = y_clean

X_train_leaky = X_leaky.iloc[train_idx]
X_test_leaky = X_leaky.iloc[test_idx]

leaky_model = LogisticRegression(max_iter=1000)

# Target column is already numeric, so use it directly
leaky_model.fit(
    X_train_leaky[["DELIBERATE_LEAK_TARGET"]],
    y_train_clean
)

leaky_prob = leaky_model.predict_proba(
    X_test_leaky[["DELIBERATE_LEAK_TARGET"]]
)[:, 1]

leaky_auc = roc_auc_score(y_test_clean, leaky_prob)

print("===== DELIBERATE LEAKAGE TEST =====")
print("Clean model ROC-AUC:", round(clean_auc, 4))
print("Leaky model ROC-AUC:", round(leaky_auc, 4))
print("AUC increase:", round(leaky_auc - clean_auc, 4))

===== DELIBERATE LEAKAGE TEST =====
Clean model ROC-AUC: 0.6157
Leaky model ROC-AUC: 1.0
AUC increase: 0.3843


### Deliberate leakage test

As a sanity check, I deliberately added the target itself as a model feature and evaluated it using the same client-grouped split.

| Model                    | ROC-AUC |
| ------------------------ | ------: |
| Clean model              |  0.6157 |
| Deliberately leaky model |  1.0000 |
| Increase                 | +0.3843 |

The deliberately leaked model achieved a ROC-AUC of 1.0000, while the clean model achieved 0.6157. This controlled experiment demonstrates how strongly target-derived information can inflate evaluation results.

The leaked feature was used only for this audit experiment and was not included in the actual model. The production feature set continues to exclude `trend_direction` and `trend_pct`.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### 4. Claim rewrite

#### Claim that would be too strong

> The model accurately predicts which content will decline in the future.

This claim goes beyond the current evaluation because the target is derived from an observed 30-day versus previous-30-day impression trend, and the current features include 90-day aggregates that can overlap the periods used to construct that label.

#### Evidence-supported claim

> On a client-grouped holdout containing 7 unseen clients, the model achieved ROC-AUC 0.6157, with precision of 0.70 at the top 20 and 0.72 at the top 50. These results indicate directional ranking signal for the observed declining-label task and may support content-prioritization decisions.

#### Important limitation

> A random row split produced higher measured performance (ROC-AUC 0.7106; P@50 0.92), but 31 clients were shared between training and test data. Therefore, the random-split results may provide an optimistic estimate of performance for unseen clients.

#### Safer interpretation

> The current evidence supports using the model as a ranking or decision-support tool for the observed declining-label task. It does not establish reliable future prediction or causal effects.


In [69]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# SECTION 4 — REAL FAILURE EXAMPLES


# Create an audit dataframe from the honest grouped test set
failure_df = df.iloc[test_idx].copy()

failure_df["model_score"] = prob_grouped
failure_df["actual_label"] = y_test_g.values


# False positives
# High model score but actual label = 0

false_positives = (
    failure_df[failure_df["actual_label"] == 0]
    .sort_values("model_score", ascending=False)
)


# False negatives
# Actual label = 1 but relatively low model score

false_negatives = (
    failure_df[failure_df["actual_label"] == 1]
    .sort_values("model_score", ascending=True)
)

display_columns = [
    "content_id",
    "client_id",
    "model_score",
    "actual_label",
    "content_type",
    "main_intent",
    "content_age_days",
    "days_since_last_update",
    "log_impressions_90d",
    "log_clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate"
]

print("===== TOP FALSE POSITIVES =====")
display(false_positives[display_columns].head(10))

print("\n===== TOP FALSE NEGATIVES =====")
display(false_negatives[display_columns].head(10))

===== TOP FALSE POSITIVES =====


,content_id,client_id,model_score,actual_label,content_type,main_intent,content_age_days,days_since_last_update,log_impressions_90d,log_clicks_90d,ctr,avg_position,engagement_rate
26614,content_7be5f150dc65,client_f369cb89fc,0.954100,0,keyword article,informational,96,20,5.673323,0.000000,0.00,5.9,0.00
20736,content_41baf0722ad9,client_8527a891e2,0.942681,0,keyword article,informational,275,104,8.044305,0.000000,0.00,12.8,0.00
12869,content_5d5653c4eb4f,client_4e07408562,0.932321,0,keyword article,informational,421,7,9.622582,0.000000,0.00,5.7,0.00
5477,content_3164f3076003,client_8527a891e2,0.924067,0,keyword article,informational,275,104,7.899895,0.693147,0.04,16.1,33.33
10175,content_374e795aab68,client_f369cb89fc,0.920298,0,keyword article,commercial,181,20,5.463832,1.098612,0.85,31.0,0.00
8139,content_7fa63804b8f1,client_4e07408562,0.919012,0,keyword article,transactional,300,104,9.019059,1.386294,0.04,7.3,0.00
27993,content_26d48a980581,client_f369cb89fc,0.916935,0,keyword article,informational,106,106,7.144407,0.000000,0.00,4.6,0.00
10136,content_df71843dcd17,client_8527a891e2,0.915777,0,keyword article,informational,237,103,10.215923,0.000000,0.00,76.4,0.00
10080,content_35d63627bf3e,client_8527a891e2,0.914118,0,keyword article,commercial,238,103,7.330405,0.000000,0.00,32.6,0.00
12332,content_4d9f36001f06,client_8527a891e2,0.912853,0,keyword article,informational,275,104,8.122668,0.693147,0.03,13.2,33.33



===== TOP FALSE NEGATIVES =====


,content_id,client_id,model_score,actual_label,content_type,main_intent,content_age_days,days_since_last_update,log_impressions_90d,log_clicks_90d,ctr,avg_position,engagement_rate
8407,content_d1e915d03c28,client_4e07408562,0.063800,1,keyword article,informational,537,104,1.098612,0.0,0.0,45.0,0.0
29158,content_e18144cbd19d,client_4e07408562,0.066531,1,keyword article,informational,545,20,1.386294,0.0,0.0,2.0,0.0
17690,content_c268b1716236,client_e629fa6598,0.079421,1,keyword article,commercial,502,20,1.386294,0.0,0.0,41.7,0.0
18655,content_e9bc92689a75,client_8527a891e2,0.103165,1,keyword article,informational,310,20,0.693147,0.0,0.0,45.0,0.0
23511,content_4de8c62603bf,client_e629fa6598,0.108974,1,keyword article,transactional,460,20,2.397895,0.0,0.0,43.2,0.0
11059,content_b52441bdc93f,client_e629fa6598,0.113026,1,keyword article,commercial,460,20,1.945910,0.0,0.0,30.3,0.0
11851,content_b0a5c92e100b,client_e629fa6598,0.116087,1,keyword article,transactional,494,20,0.693147,0.0,0.0,11.0,0.0
7042,content_b3623d22db24,client_e629fa6598,0.116420,1,keyword article,commercial,502,20,2.397895,0.0,0.0,36.3,0.0
16635,content_69e05c36b3c7,client_e629fa6598,0.119862,1,keyword article,commercial,460,20,1.386294,0.0,0.0,7.0,0.0
13160,content_dfbd76532fb8,client_8527a891e2,0.120941,1,keyword article,informational,348,20,1.945910,0.0,0.0,60.2,0.0


In [70]:

# SECTION 4 — FAILURE EXAMPLES WITH SIMPLE PATTERN CHECKS


# Top 5 false positives and false negatives
fp_examples = false_positives[display_columns].head(5).copy()
fn_examples = false_negatives[display_columns].head(5).copy()

print("===== TOP 5 FALSE POSITIVES =====")
display(fp_examples)

print("\n===== TOP 5 FALSE NEGATIVES =====")
display(fn_examples)


# Compare basic characteristics of FP vs FN vs all test data


failure_summary = pd.DataFrame({
    "Group": ["All honest test", "False positives", "False negatives"],
    "Rows": [
        len(failure_df),
        len(false_positives),
        len(false_negatives)
    ],
    "Mean score": [
        failure_df["model_score"].mean(),
        false_positives["model_score"].mean(),
        false_negatives["model_score"].mean()
    ],
    "Declining rate": [
        failure_df["actual_label"].mean(),
        false_positives["actual_label"].mean(),
        false_negatives["actual_label"].mean()
    ],
    "Mean content age": [
        failure_df["content_age_days"].mean(),
        false_positives["content_age_days"].mean(),
        false_negatives["content_age_days"].mean()
    ],
    "Mean CTR": [
        failure_df["ctr"].mean(),
        false_positives["ctr"].mean(),
        false_negatives["ctr"].mean()
    ]
})

print("\n===== FAILURE SUMMARY =====")
display(failure_summary.round(4))

===== TOP 5 FALSE POSITIVES =====


,content_id,client_id,model_score,actual_label,content_type,main_intent,content_age_days,days_since_last_update,log_impressions_90d,log_clicks_90d,ctr,avg_position,engagement_rate
26614,content_7be5f150dc65,client_f369cb89fc,0.954100,0,keyword article,informational,96,20,5.673323,0.000000,0.00,5.9,0.00
20736,content_41baf0722ad9,client_8527a891e2,0.942681,0,keyword article,informational,275,104,8.044305,0.000000,0.00,12.8,0.00
12869,content_5d5653c4eb4f,client_4e07408562,0.932321,0,keyword article,informational,421,7,9.622582,0.000000,0.00,5.7,0.00
5477,content_3164f3076003,client_8527a891e2,0.924067,0,keyword article,informational,275,104,7.899895,0.693147,0.04,16.1,33.33
10175,content_374e795aab68,client_f369cb89fc,0.920298,0,keyword article,commercial,181,20,5.463832,1.098612,0.85,31.0,0.00



===== TOP 5 FALSE NEGATIVES =====


,content_id,client_id,model_score,actual_label,content_type,main_intent,content_age_days,days_since_last_update,log_impressions_90d,log_clicks_90d,ctr,avg_position,engagement_rate
8407,content_d1e915d03c28,client_4e07408562,0.063800,1,keyword article,informational,537,104,1.098612,0.0,0.0,45.0,0.0
29158,content_e18144cbd19d,client_4e07408562,0.066531,1,keyword article,informational,545,20,1.386294,0.0,0.0,2.0,0.0
17690,content_c268b1716236,client_e629fa6598,0.079421,1,keyword article,commercial,502,20,1.386294,0.0,0.0,41.7,0.0
18655,content_e9bc92689a75,client_8527a891e2,0.103165,1,keyword article,informational,310,20,0.693147,0.0,0.0,45.0,0.0
23511,content_4de8c62603bf,client_e629fa6598,0.108974,1,keyword article,transactional,460,20,2.397895,0.0,0.0,43.2,0.0



===== FAILURE SUMMARY =====


,Group,Rows,Mean score,Declining rate,Mean content age,Mean CTR
0,All honest test,6163,0.5566,0.511,293.2569,0.2901
1,False positives,3014,0.5171,0.000,306.9738,0.3138
2,False negatives,3149,0.5945,1.000,280.1280,0.2674


In [71]:

# FAILURE EXAMPLE INTERPRETATION CHECK


print("False positives available:", len(false_positives))
print("False negatives available:", len(false_negatives))

print("\nFalse-positive content types:")
print(false_positives["content_type"].value_counts().head())

print("\nFalse-negative content types:")
print(false_negatives["content_type"].value_counts().head())

print("\nFalse-positive intents:")
print(false_positives["main_intent"].value_counts().head())

print("\nFalse-negative intents:")
print(false_negatives["main_intent"].value_counts().head())

False positives available: 3014
False negatives available: 3149

False-positive content types:
content_type
keyword article    3014
Name: count, dtype: int64

False-negative content types:
content_type
keyword article    3149
Name: count, dtype: int64

False-positive intents:
main_intent
informational    2015
commercial        493
transactional     492
unknown             9
navigational        5
Name: count, dtype: int64

False-negative intents:
main_intent
informational    1985
transactional     660
commercial        493
unknown             9
navigational        2
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Audit conclusion

The validation audit shows that evaluation methodology materially affects the measured model performance. The random row split produced higher metrics, but it allowed 31 clients to appear in both training and test data. The client-grouped evaluation removed this overlap and produced ROC-AUC 0.6157, P@20 0.70, P@50 0.72, and P@100 0.70.

The leakage audit confirmed that the direct label-source fields `trend_direction` and `trend_pct` were excluded from the model. The deliberate leakage experiment demonstrated the potential inflation caused by target-derived information, with ROC-AUC increasing from 0.6157 to 1.0000 when the target was deliberately added as a feature.

The current evidence supports describing the model as a ranking or decision-support tool for the observed declining-label task. It does not establish reliable future prediction or causal impact. A future version should use a strictly time-aware feature window and future outcome window to evaluate genuine forward prediction.
